# QCM-RF Module
This tutorial uses the QCM-RF module and a Marki M20004LA mixer. The outputs O1 and O2 of the QCM are connected to the L and R ports of the mixer, and the mixer output port (labelled I) is connected to an oscilloscope. The purpose of this tutorial is to get a basic output from the QCM-RF and verify that modulation is working correctly. The two sequencers being used each take a square pulse and modulate it to the NCO frequency. The local oscillator (LO) then modulates the signal again before outputting the resulting waveform. The two outputs from qblox then go into the mixer which outputs the difference in frequency between the two LOs, which can be seen on the oscilloscope. It is worth noting that, you won't see a pure sinusoid coming from the mixer because the modulation/demodulation introduces other harmonics which alter the signal. This is easily seen if you look at the frequency specturm of the mixer's output. A mixer is being used to downconvert the output frequencies low enough so that signals can be seen on an oscilloscope, since the minimum ouput frequency from the QCM-RF is 2GHz.\
\
Note: All signals must be modulated by the NCO and LO, to the best of our knowledge, you can not use the QCM-RF to send out unmodulated signals.\
\
Authors: Noah Stieler, Luke Dyer

In [1]:
import scipy

from qblox_instruments import Cluster
from qblox_instruments.types import InstrumentType

In [2]:
#Scan for clusters
!qblox-pnp list

Devices:
 - 192.168.137.2: cluster_mm 0.6.2 with name "cluster-mm" and serial number 00015_2251_003


In [3]:
#Connect to the cluster and select the QCM-RF module.
cluster = Cluster("cluster_mm", "192.168.137.2")
moduleQCM = None

modules = [mod for mod in cluster.modules if mod.present()]
for mod in modules:
	print(str(mod) + " type=" + str(mod.module_type) + " is_rf_type=" + str(mod.is_rf_type))

cluster.reset()
cluster.get_system_state()

moduleQCM = modules[2]

#Configure the QCM-RF module
moduleQCM.disconnect_outputs()

#When using the mixer, it is important that both outputs are synchronized,
#otherwise a random phase offset is introduced between the signals which 
#may cause them to interfere destructively.
moduleQCM.sequencer0.sync_en(True)
moduleQCM.sequencer1.sync_en(True)

#Set which sequencers are on which outputs
moduleQCM.sequencer0.connect_out0(True)
moduleQCM.sequencer1.connect_out1(True)

#Remember that both NCO and LO modulation must be enabled and set
#in order for the module to work.

#Enable amd set NCO modulation
moduleQCM.sequencer0.mod_en_awg(True)
moduleQCM.sequencer1.mod_en_awg(True)
moduleQCM.sequencer0.nco_freq(20e6)
moduleQCM.sequencer1.nco_freq(20e6)

#Enable and set LO modulation
moduleQCM.out0_lo_en(True)
moduleQCM.out0_lo_freq(4e9)
moduleQCM.out1_lo_en(True)
moduleQCM.out1_lo_freq(4.2e9)


<QcmQrm: cluster_mm_module2 of Cluster: cluster_mm> type=QCM is_rf_type=False
<QcmQrm: cluster_mm_module4 of Cluster: cluster_mm> type=QRM is_rf_type=False
<QcmQrm: cluster_mm_module6 of Cluster: cluster_mm> type=QCM is_rf_type=True


In [4]:
#This sequence program plays a modulated square pulse.
#VERY IMPORTANT:
#	set_mark command must be used in order to enable QCM-RF output.
#	the parameter is a 4 bit number (so 0-15).
#   bit indices 0 and 1 enable either O1 or O2 and
#	bit indices 2 and 3 enable either marker output 2 or marker output 1
#
#Thus, set_mark 15 corresponds to set_mark 1111 in binary which means
#O1, O2, and both marker outputs are enabled.
#
#See part 6 of the following documentation page for more info:
#https://qblox-qblox-instruments.readthedocs-hosted.com/en/main/cluster/qcm_rf.html#qcm-rf-marker-output-channels
sequenceProg = f"""
	wait_sync	4
	set_mrk		15			
	upd_param	4
	play		0,0,16384
	set_mrk		0
	upd_param	4
	stop
"""

#Standard waveform, and acquisition specification and upload to the sequencers.
waveforms = {
	"square": {"data": [1.0 for i in range(16000)], "index": 0},
	#Uncomment this and comment out the square pulse to use the gaussian.
	#"gaussian": {
    #   "data": scipy.signal.windows.gaussian(16000, std=0.12 * 16000).tolist(),
    #   "index": 0,
    #},
}
acquisitions = {
	"acq": {"num_bins": 1, "index": 0}	
}
sequence = {
	"waveforms": waveforms,
	"weights": {},
	"acquisitions": acquisitions,
	"program": sequenceProg,
}

moduleQCM.sequencer0.sequence(sequence)
moduleQCM.sequencer1.sequence(sequence)

In [5]:
#Play the sequences.
moduleQCM.arm_sequencer(0)
moduleQCM.arm_sequencer(1)

moduleQCM.start_sequencer()

print(moduleQCM.get_sequencer_state(0))
print(moduleQCM.get_sequencer_state(1))

Status: STOPPED, Flags: NONE
Status: STOPPED, Flags: NONE
